In [3]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Load dataset
data = pd.read_csv("../tilak/Downloads/ner_datasetreference.csv",encoding="latin1")

# Fill missing sentence ids
data['Sentence #'] = data['Sentence #'].fillna(method='ffill')

# Group words and tags by sentence
sentences = data.groupby('Sentence #')['Word'].apply(list).values
tags = data.groupby('Sentence #')['Tag'].apply(list).values

# Build vocab
words = list(set(data['Word']))
tags_list = list(set(data['Tag']))

word2idx = {w:i+1 for i,w in enumerate(words)}
tag2idx = {t:i for i,t in enumerate(tags_list)}

# Convert to numbers
X = [[word2idx[w] for w in s] for s in sentences]
y = [[tag2idx[t] for t in s] for s in tags]

# Padding
max_len = 30
X = pad_sequences(X, maxlen=max_len, padding='post')
y = pad_sequences(y, maxlen=max_len, padding='post')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Model
model = Sequential()
model.add(Embedding(len(word2idx)+1, 64, input_length=max_len))
model.add(LSTM(64, return_sequences=True))
model.add(Dense(len(tag2idx), activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X_train, np.expand_dims(y_train, -1), epochs=3, batch_size=32)

# Test
loss, acc = model.evaluate(X_test, np.expand_dims(y_test, -1))
print("Accuracy:", acc)

C:\Users\tilak\AppData\Local\Temp\ipykernel_9704\2425989985.py:12: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['Sentence #'] = data['Sentence #'].fillna(method='ffill')
C:\Users\tilak\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 72s 53ms/step - accuracy: 0.9418 - loss: 0.2748
Epoch 2/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 64s 53ms/step - accuracy: 0.9749 - loss: 0.0858
Epoch 3/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 80s 51ms/step - accuracy: 0.9790 - loss: 0.0658
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.9736 - loss: 0.0858
Accuracy: 0.9736273288726807


In [8]:
# reverse maps
idx2word = {i:w for w,i in word2idx.items()}
idx2tag = {i:t for t,i in tag2idx.items()}

# sample
x = X_test[0]

# FIX: add batch dimension
pred = model.predict(np.array([x]))

# get result
pred = pred.argmax(axis=-1)[0]

print("\nPrediction:")
for i in range(len(x)):
    if x[i] != 0:
        print(idx2word[x[i]], "->", idx2tag[pred[i]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step

Prediction:
They -> O
say -> O
the -> O
masked -> O
police -> O
surrounded -> O
a -> O
municipal -> O
government -> O
building -> O
, -> O
taking -> O
positions -> O
on -> O
the -> O
roof -> O
and -> O
firing -> O
in -> O
the -> O
air -> O
. -> O
